In [ ]:
# In[1]:

import pandas as pd

# Load CSVs
metric_df = pd.read_csv('metric.csv')
trace_df = pd.read_csv('trace.csv')
log_df = pd.read_csv('log.csv')
error_df = pd.read_csv('error_logs.csv')

# Parse timestamps to UTC datetimes
for df in (metric_df, trace_df, log_df, error_df):
    if 'timestamp' in df.columns:
        df['timestamp'] = pd.to_datetime(df['timestamp'], unit='s', utc=True)

# Ensure numeric 'value' columns
metric_df['value'] = pd.to_numeric(metric_df['value'], errors='coerce')
trace_df['value'] = pd.to_numeric(trace_df['value'], errors='coerce')
log_df['value'] = pd.to_numeric(log_df['value'], errors='coerce')

# --- METRIC aggregates ---
# 1) cmdb_id counts (top 20)
metric_cmdb_counts = (
    metric_df.groupby('cmdb_id', dropna=False)
    .size()
    .reset_index(name='row_count')
    .sort_values('row_count', ascending=False)
    .head(20)
)

# 2) kpi_name counts (top 20)
metric_kpi_counts = (
    metric_df.groupby('kpi_name', dropna=False)
    .size()
    .reset_index(name='row_count')
    .sort_values('row_count', ascending=False)
    .head(20)
)

# 3) global P95 per (cmdb_id, kpi_name)
metric_p95 = (
    metric_df.groupby(['cmdb_id', 'kpi_name'], dropna=False)['value']
    .agg(p95_value=lambda x: x.quantile(0.95), row_count='size')
    .reset_index()
    .sort_values('p95_value', ascending=False)
)
metric_p95_top20 = metric_p95.head(20)

# --- TRACE aggregates ---
trace_cmdb_counts = (
    trace_df.groupby('cmdb_id', dropna=False)
    .size()
    .reset_index(name='row_count')
    .sort_values('row_count', ascending=False)
    .head(20)
)

trace_name_counts = (
    trace_df.groupby('trace_name', dropna=False)
    .size()
    .reset_index(name='row_count')
    .sort_values('row_count', ascending=False)
    .head(20)
)

trace_p95 = (
    trace_df.groupby(['cmdb_id', 'trace_name'], dropna=False)['value']
    .agg(p95_value=lambda x: x.quantile(0.95), row_count='size')
    .reset_index()
    .sort_values('p95_value', ascending=False)
)
trace_p95_top20 = trace_p95.head(20)

# --- LOG aggregates ---
log_cmdb_counts = (
    log_df.groupby('cmdb_id', dropna=False)
    .size()
    .reset_index(name='row_count')
    .sort_values('row_count', ascending=False)
    .head(20)
)

log_name_counts = (
    log_df.groupby('log_name', dropna=False)
    .size()
    .reset_index(name='row_count')
    .sort_values('row_count', ascending=False)
    .head(20)
)

log_p95 = (
    log_df.groupby(['cmdb_id', 'log_name'], dropna=False)['value']
    .agg(p95_value=lambda x: x.quantile(0.95), row_count='size')
    .reset_index()
    .sort_values('p95_value', ascending=False)
)
log_p95_top20 = log_p95.head(20)

# --- ERROR LOGS: unique cmdb_id and earliest 10 sample rows ---
error_cmdb_counts = (
    error_df.groupby('cmdb_id', dropna=False)
    .size()
    .reset_index(name='row_count')
    .sort_values('row_count', ascending=False)
)
# earliest 10 rows (compact)
error_sample = error_df.sort_values('timestamp').head(10)[['timestamp', 'cmdb_id', 'message']]

# Display compact aggregated results (multiple variables separated by commas)
metric_cmdb_counts, metric_kpi_counts, metric_p95_top20, \
trace_cmdb_counts, trace_name_counts, trace_p95_top20, \
log_cmdb_counts, log_name_counts, log_p95_top20, \
error_cmdb_counts, error_sample

```
Out[1]:
```
summary = (
    "Summary of telemetry aggregates (global / full data):\n\n"
    "Metrics (metric.csv):\n"
    "- Top cmdb_id by row count: frontend (200 rows); several services at 175 rows each "
    "(adservice, currencyservice, cartservice, recommendationservice, emailservice, productcatalogservice); "
    "checkoutservice/shippingservice/paymentservice (~150 rows); redis (100), frontend-external (50).\n"
    "- Top KPI names by row count: cpu, workload, socket, mem (each 275 rows); latency-50 and latency-90 (250 rows); "
    "diskio (200), error (50).\n"
    "- Top metric P95 values (by p95_value): memory-related P95s dominate. Highest is adservice mem (~1.09e8), "
    "followed by recommendationservice mem (~4.56e7), currencyservice mem (~4.54e7), cartservice mem (~4.46e7), "
    "emailservice mem (~4.38e7). Each of these P95 rows corresponds to series with 25 samples.\n\n"
    "Traces (trace.csv):\n"
    "- Top cmdb_id by row count: checkoutservice (800 rows), frontendservice (700), root (700); several other services with fewer rows.\n"
    "- Top trace feature names: many 'trace.from_root.*' and 'trace.from_frontendservice.*' series, plus per-destination series "
    "(e.g., trace.to_productcatalogservice.*, trace.to_recommendationservice.*).\n"
    "- Top trace P95 values: highest P95s are for trace row_count series on frontendservice (e.g., trace.to_frontendservice.row_count "
    "and trace.from_frontendservice.row_count ~8153.8), then various service-to-service row_count P95s (values in the thousands down to hundreds). "
    "All top trace P95 rows shown have 25 samples.\n\n"
    "Logs (log.csv):\n"
    "- cmdb_id row counts are small and similar across services (many at 50 rows; redis has 10 rows).\n"
    "- log feature names: log.error_count and log.row_count (each 230 rows overall).\n"
    "- Top log P95 values: frontend log.row_count has the largest P95 (~2828.2), followed by currencyservice log.row_count (~2548.4), cartservice (~810.8), etc. "
    "log.error_count P95 is 0.0 for the listed services (no high error_count observed in the global series).\n\n"
    "Error logs (error_logs.csv):\n"
    "- No error_log rows or unique cmdb_id samples were returned in the earliest-10 sample (empty result).\n\n"
    "Overall takeaway: memory (mem) P95s stand out in the metric data with very large values for several services (adservice highest). "
    "Trace row_count series show very high P95 counts for frontend-related traces. Log error_count is globally zero in the available data, "
    "and there were no error log samples returned. These are global, full-data summaries (P95 computed across the entire series before any time filtering)."
)
summary

The original code execution output of IPython Kernel is also provided below for reference:

(                  cmdb_id  row_count
5                frontend        200
0               adservice        175
3         currencyservice        175
1             cartservice        175
9   recommendationservice        175
4            emailservice        175
8   productcatalogservice        175
2         checkoutservice        150
11        shippingservice        150
7          paymentservice        150
10                  redis        100
6       frontend-external         50,      kpi_name  row_count
0         cpu        275
7    workload        275
6      socket        275
5         mem        275
3  latency-50        250
4  latency-90        250
1      diskio        200
2       error         50,                   cmdb_id  kpi_name     p95_value  row_count
4               adservice       mem  1.094921e+08         25
61  recommendationservice       mem  4.561823e+07         25
24        currencyservice       mem  4.541492e+07         25
11            cartservice       mem  4.457980e+07         25
31           emailservice       mem  4.375654e+07         25
47         paymentservice       mem  3.894953e+07         25
39               frontend       mem  1.608219e+07         25
17        checkoutservice       mem  1.200431e+07         25
54  productcatalogservice       mem  1.130819e+07         25
71        shippingservice       mem  8.952928e+06         25
66                  redis       mem  2.998217e+06         25
1               adservice    diskio  7.368951e+04         25
65                  redis    diskio  1.385009e+04         25
56  productcatalogservice  workload  1.481775e+02         25
26        currencyservice  workload  8.342830e+01         25
20        currencyservice       cpu  3.324464e+01         25
40               frontend    socket  2.700000e+01         25
13            cartservice  workload  2.636226e+01         25
63  recommendationservice  workload  1.924015e+01         25
6               adservice  workload  1.573318e+01         25,                  cmdb_id  row_count
0        checkoutservice        800
3        frontendservice        700
7                   root        700
6  recommendationservice        500
5  productcatalogservice        400
1        currencyservice        300
2           emailservice        200
4         paymentservice        200,                                       trace_name  row_count
15                     trace.from_root.row_count        175
14                    trace.from_root.error_rate        175
13                  trace.from_root.duration_p95        175
12                 trace.from_root.duration_mean        175
4       trace.from_frontendservice.duration_mean        125
5        trace.from_frontendservice.duration_p95        125
6          trace.from_frontendservice.error_rate        125
7           trace.from_frontendservice.row_count        125
0       trace.from_checkoutservice.duration_mean        125
1        trace.from_checkoutservice.duration_p95        125
2          trace.from_checkoutservice.error_rate        125
3           trace.from_checkoutservice.row_count        125
36  trace.to_productcatalogservice.duration_mean        100
37   trace.to_productcatalogservice.duration_p95        100
38     trace.to_productcatalogservice.error_rate        100
39      trace.to_productcatalogservice.row_count        100
43      trace.to_recommendationservice.row_count         75
42     trace.to_recommendationservice.error_rate         75
41   trace.to_recommendationservice.duration_p95         75
40  trace.to_recommendationservice.duration_mean         75,                    cmdb_id                                  trace_name  p95_value  row_count
71         frontendservice          trace.to_frontendservice.row_count     8153.8         25
55         frontendservice        trace.from_frontendservice.row_count     8153.8         25
95   productcatalogservice        trace.from_frontendservice.row_count     3787.4         25
75         frontendservice    trace.to_productcatalogservice.row_count     3787.4         25
39         currencyservice        trace.from_frontendservice.row_count     2438.2         25
67         frontendservice          trace.to_currencyservice.row_count     2438.2         25
59         frontendservice                   trace.from_root.row_count      858.4         25
139                   root          trace.to_frontendservice.row_count      858.4         25
119  recommendationservice    trace.to_productcatalogservice.row_count      578.2         25
107  recommendationservice        trace.from_frontendservice.row_count      578.2         25
111  recommendationservice  trace.from_recommendationservice.row_count      578.2         25
79         frontendservice    trace.to_recommendationservice.row_count      578.2         25
123  recommendationservice    trace.to_recommendationservice.row_count      578.2         25
99   productcatalogservice  trace.from_recommendationservice.row_count      578.2         25
3          checkoutservice        trace.from_checkoutservice.row_count      311.8         25
15         checkoutservice          trace.to_checkoutservice.row_count      311.8         25
11         checkoutservice                   trace.from_root.row_count      154.8         25
127                   root          trace.to_checkoutservice.row_count      154.8         25
35         currencyservice        trace.from_checkoutservice.row_count      130.8         25
19         checkoutservice          trace.to_currencyservice.row_count      130.8         25,                  cmdb_id  row_count
0              adservice         50
1            cartservice         50
2        checkoutservice         50
3        currencyservice         50
4           emailservice         50
5               frontend         50
6         paymentservice         50
7  recommendationservice         50
9        shippingservice         50
8                  redis         10,           log_name  row_count
0  log.error_count        230
1    log.row_count        230,                   cmdb_id         log_name  p95_value  row_count
11               frontend    log.row_count     2828.2         25
7         currencyservice    log.row_count     2548.4         25
3             cartservice    log.row_count      810.8         25
15  recommendationservice    log.row_count      578.2         25
19        shippingservice    log.row_count      491.2         25
1               adservice    log.row_count      470.8         25
5         checkoutservice    log.row_count      110.4         25
13         paymentservice    log.row_count       73.6         25
9            emailservice    log.row_count       36.8         25
17                  redis    log.row_count        5.0          5
2             cartservice  log.error_count        0.0         25
0               adservice  log.error_count        0.0         25
10               frontend  log.error_count        0.0         25
8            emailservice  log.error_count        0.0         25
4         checkoutservice  log.error_count        0.0         25
6         currencyservice  log.error_count        0.0         25
14  recommendationservice  log.error_count        0.0         25
12         paymentservice  log.error_count        0.0         25
16                  redis  log.error_count        0.0          5
18        shippingservice  log.error_count        0.0         25, Empty DataFrame
Columns: [cmdb_id, row_count]
Index: [], Empty DataFrame
Columns: [timestamp, cmdb_id, message]
Index: [])```
```

In [ ]:
# In[2]:

import pandas as pd
import numpy as np
from pandas.api.types import is_datetime64_any_dtype, is_integer_dtype

# Reuse existing metric_df if present; otherwise load
try:
    metric_df
except NameError:
    metric_df = pd.read_csv('metric.csv')

# Ensure timestamp is parsed to UTC datetime (only parse if not already datetime)
if not is_datetime64_any_dtype(metric_df['timestamp']):
    # If timestamps are integers (seconds), parse with unit='s'
    if is_integer_dtype(metric_df['timestamp']):
        metric_df['timestamp'] = pd.to_datetime(metric_df['timestamp'], unit='s', utc=True)
    else:
        metric_df['timestamp'] = pd.to_datetime(metric_df['timestamp'], utc=True)

# Ensure numeric values
metric_df['value'] = pd.to_numeric(metric_df['value'], errors='coerce')

# 1) Compute global P95 per (cmdb_id, kpi_name) using the full metric_df (global threshold before filtering)
global_p95_df = (
    metric_df.groupby(['cmdb_id', 'kpi_name'], dropna=False)['value']
    .agg(global_p95=lambda x: x.quantile(0.95))
    .reset_index()
)

# 2) Define incident window and filter metric_df to that window
start_ts = pd.to_datetime('2024-01-20 07:13:22', utc=True)
end_ts   = pd.to_datetime('2024-01-20 07:43:22', utc=True)
window_df = metric_df[(metric_df['timestamp'] >= start_ts) & (metric_df['timestamp'] <= end_ts)].copy()

# 3) Merge global_p95 into window rows to label anomalies (value > global_p95)
# If a (cmdb_id,kpi_name) exists in window but not in global_p95 (unlikely), we'll keep global_p95 as NaN
window_with_p95 = window_df.merge(global_p95_df, on=['cmdb_id','kpi_name'], how='left')

# Mark anomalies where value > global_p95 (NaN global_p95 -> False)
window_with_p95['is_anomaly'] = (window_with_p95['value'] > window_with_p95['global_p95']).fillna(False)

# Aggregate per (cmdb_id, kpi_name) within the window
agg_window = (
    window_with_p95.groupby(['cmdb_id','kpi_name'], dropna=False)
    .agg(
        row_count_in_window=('value', 'size'),
        anomaly_count=('is_anomaly', 'sum'),
        earliest_anomaly_timestamp=('timestamp', lambda x: x[window_with_p95.loc[x.index, 'is_anomaly']].min() if x.size>0 else pd.NaT),
        max_value_in_window=('value', 'max')
    )
    .reset_index()
)

# 3b) Ensure we include series that exist globally but had zero rows in the window
# Left-join global_p95_df with agg_window
summary_df = global_p95_df.merge(agg_window, on=['cmdb_id','kpi_name'], how='left')

# Fill NaNs for window-derived metrics: zeros for counts, NaT for timestamps, NaN for max_value
summary_df['row_count_in_window'] = summary_df['row_count_in_window'].fillna(0).astype(int)
summary_df['anomaly_count'] = summary_df['anomaly_count'].fillna(0).astype(int)
summary_df['earliest_anomaly_timestamp'] = pd.to_datetime(summary_df['earliest_anomaly_timestamp'])
# Format earliest_anomaly_timestamp as ISO string (UTC) for display; keep NaT as None
summary_df['earliest_anomaly_timestamp'] = summary_df['earliest_anomaly_timestamp'].dt.strftime('%Y-%m-%dT%H:%M:%SZ')
summary_df.loc[summary_df['earliest_anomaly_timestamp'].isna(), 'earliest_anomaly_timestamp'] = None
# max_value_in_window remains numeric; keep NaN for missing
# Compute severity difference for sorting (max_value - global_p95)
summary_df['severity_diff'] = summary_df['max_value_in_window'] - summary_df['global_p95']
# Replace NaN severity_diff with -inf to push empty series to bottom when sorting
summary_df['severity_diff'] = summary_df['severity_diff'].fillna(-np.inf)

# 4) Sort by anomaly_count desc then by severity_diff desc and limit to top 20
result_top20 = (
    summary_df.sort_values(['anomaly_count', 'severity_diff'], ascending=[False, False])
    .loc[:, ['cmdb_id','kpi_name','global_p95','row_count_in_window','anomaly_count','earliest_anomaly_timestamp','max_value_in_window']]
    .head(20)
)

# Short summary: which cmdb_id have any anomalies in the window and total anomaly counts per cmdb_id (sorted)
anomaly_by_cmdb = (
    summary_df.groupby('cmdb_id', dropna=False)['anomaly_count']
    .sum()
    .reset_index(name='total_anomalies')
    .query('total_anomalies > 0')
    .sort_values('total_anomalies', ascending=False)
)

if anomaly_by_cmdb.shape[0] == 0:
    summary_line = "No anomalies detected in the given window for any cmdb_id."
else:
    # compact textual summary
    parts = [f"{row.cmdb_id}:{row.total_anomalies}" for row in anomaly_by_cmdb.itertuples()]
    summary_line = "Anomalies by cmdb_id (total counts): " + ", ".join(parts)

# Display compact results: aggregated table and the short summary line
result_top20, anomaly_by_cmdb, summary_line

```
Out[2]:
```
summary = (
    "Summary (window 2024-01-20 07:13:22–07:43:22 UTC):\n\n"
    "1) Global P95 thresholds were computed on the full metric.csv before filtering.\n\n"
    "2) During the 30-minute window many services exceeded their global P95s. The dominant anomaly type is memory (kpi_name = mem) — multiple services show mem series with anomaly_count = 2 (examples: adservice, currencyservice, cartservice, recommendationservice, emailservice, frontend, productcatalogservice, shippingservice, checkoutservice, paymentservice, redis). Adservice has a very large global mem P95 and mem exceeded it in the window (earliest mem anomaly 2024-01-20T07:19:00Z).\n\n"
    "3) Other anomalous KPIs in the top results include diskio (adservice, redis), cpu (currencyservice), and workload series (productcatalogservice, recommendationservice, frontend, frontend-external, shippingservice). For each top series the table shows global_p95, 25 rows in the window, anomaly_count = 2, earliest anomaly timestamps in the ~07:18–07:38Z range, and the max values in-window slightly above the global P95.\n\n"
    "4) Aggregate anomaly totals by cmdb_id (sorted): adservice:13, currencyservice:11, frontend:11, cartservice:10, checkoutservice:10, emailservice:10, paymentservice:10, productcatalogservice:10, shippingservice:10, recommendationservice:10, redis:6, frontend-external:2.\n\n"
    "Bottom line: memory-related spikes occurred across many services during the window (adservice stands out with the highest mem P95 and multiple mem anomalies)."
)
summary

The original code execution output of IPython Kernel is also provided below for reference:

(                  cmdb_id  kpi_name    global_p95  row_count_in_window  anomaly_count earliest_anomaly_timestamp  max_value_in_window
47         paymentservice       mem  3.894953e+07                   25              2       2024-01-20T07:23:00Z         3.927586e+07
17        checkoutservice       mem  1.200431e+07                   25              2       2024-01-20T07:23:00Z         1.222117e+07
4               adservice       mem  1.094921e+08                   25              2       2024-01-20T07:19:00Z         1.096215e+08
54  productcatalogservice       mem  1.130819e+07                   25              2       2024-01-20T07:38:00Z         1.143637e+07
24        currencyservice       mem  4.541492e+07                   25              2       2024-01-20T07:36:00Z         4.552335e+07
1               adservice    diskio  7.368951e+04                   25              2       2024-01-20T07:30:00Z         1.063900e+05
39               frontend       mem  1.608219e+07                   25              2       2024-01-20T07:26:00Z         1.610370e+07
11            cartservice       mem  4.457980e+07                   25              2       2024-01-20T07:18:00Z         4.460107e+07
61  recommendationservice       mem  4.561823e+07                   25              2       2024-01-20T07:34:00Z         4.562285e+07
71        shippingservice       mem  8.952928e+06                   25              2       2024-01-20T07:26:00Z         8.956450e+06
31           emailservice       mem  4.375654e+07                   25              2       2024-01-20T07:33:00Z         4.375818e+07
66                  redis       mem  2.998217e+06                   25              2       2024-01-20T07:21:00Z         2.999637e+06
65                  redis    diskio  1.385009e+04                   25              2       2024-01-20T07:27:00Z         1.416747e+04
56  productcatalogservice  workload  1.481775e+02                   25              2       2024-01-20T07:18:00Z         1.516533e+02
20        currencyservice       cpu  3.324464e+01                   25              2       2024-01-20T07:23:00Z         3.425242e+01
73        shippingservice  workload  8.133870e+00                   25              2       2024-01-20T07:22:00Z         8.874417e+00
63  recommendationservice  workload  1.924015e+01                   25              2       2024-01-20T07:34:00Z         1.994650e+01
26        currencyservice  workload  8.342830e+01                   25              2       2024-01-20T07:18:00Z         8.385547e+01
41               frontend  workload  1.366699e+01                   25              2       2024-01-20T07:18:00Z         1.389898e+01
43      frontend-external  workload  1.360761e+01                   25              2       2024-01-20T07:26:00Z         1.382015e+01,                   cmdb_id  total_anomalies
0               adservice               13
3         currencyservice               11
5                frontend               11
1             cartservice               10
2         checkoutservice               10
4            emailservice               10
7          paymentservice               10
8   productcatalogservice               10
11        shippingservice               10
9   recommendationservice               10
10                  redis                6
6       frontend-external                2, 'Anomalies by cmdb_id (total counts): adservice:13, currencyservice:11, frontend:11, cartservice:10, checkoutservice:10, emailservice:10, paymentservice:10, productcatalogservice:10, shippingservice:10, recommendationservice:10, redis:6, frontend-external:2')```
```

In [ ]:
# In[3]:

import pandas as pd
import numpy as np
from pandas.api.types import is_datetime64_any_dtype, is_integer_dtype

# Reuse existing trace_df if available; otherwise load
try:
    trace_df
except NameError:
    trace_df = pd.read_csv('trace.csv')

# Ensure timestamp parsed to UTC datetime (unit='s') as required
if not is_datetime64_any_dtype(trace_df['timestamp']):
    if is_integer_dtype(trace_df['timestamp']):
        trace_df['timestamp'] = pd.to_datetime(trace_df['timestamp'], unit='s', utc=True)
    else:
        trace_df['timestamp'] = pd.to_datetime(trace_df['timestamp'], utc=True)

# Ensure numeric 'value'
trace_df['value'] = pd.to_numeric(trace_df['value'], errors='coerce')

# 1) Compute global P95 and P05 per (cmdb_id, trace_name) using full data (before filtering)
global_thresh = (
    trace_df.groupby(['cmdb_id','trace_name'], dropna=False)['value']
    .agg(global_p95=lambda x: x.quantile(0.95),
         global_p05=lambda x: x.quantile(0.05))
    .reset_index()
)

# 2) Filter to incident window
start_ts = pd.to_datetime('2024-01-20 07:13:22', utc=True)
end_ts   = pd.to_datetime('2024-01-20 07:43:22', utc=True)
window_df = trace_df[(trace_df['timestamp'] >= start_ts) & (trace_df['timestamp'] <= end_ts)].copy()

# 3) Merge global thresholds into window rows (so anomalies are determined against global thresholds)
window_with_thresh = window_df.merge(global_thresh, on=['cmdb_id','trace_name'], how='left')

# Mark high/low anomalies (NaN thresholds => False)
window_with_thresh['is_high'] = (window_with_thresh['value'] > window_with_thresh['global_p95']).fillna(False)
window_with_thresh['is_low']  = (window_with_thresh['value'] < window_with_thresh['global_p05']).fillna(False)

# Aggregate counts and min/max in window per series
agg_window = (
    window_with_thresh.groupby(['cmdb_id','trace_name'], dropna=False)
    .agg(
        row_count_in_window=('value','size'),
        high_anomaly_count=('is_high','sum'),
        low_anomaly_count=('is_low','sum'),
        min_value_in_window=('value','min'),
        max_value_in_window=('value','max')
    )
    .reset_index()
)

# Earliest low anomaly timestamp per series (if any)
earliest_low = (
    window_with_thresh[window_with_thresh['is_low']]
    .groupby(['cmdb_id','trace_name'], dropna=False)['timestamp']
    .min()
    .reset_index()
    .rename(columns={'timestamp':'earliest_low_anomaly_timestamp'})
)

# Merge earliest_low into agg_window
agg_window = agg_window.merge(earliest_low, on=['cmdb_id','trace_name'], how='left')

# 3b) Ensure we include all global series (even those with zero rows in window)
summary_df = global_thresh.merge(agg_window, on=['cmdb_id','trace_name'], how='left')

# Fill missing window-derived values: zeros for counts, NaN for min/max, NaT/None for timestamps
summary_df['row_count_in_window'] = summary_df['row_count_in_window'].fillna(0).astype(int)
summary_df['high_anomaly_count'] = summary_df['high_anomaly_count'].fillna(0).astype(int)
summary_df['low_anomaly_count'] = summary_df['low_anomaly_count'].fillna(0).astype(int)
# Format earliest_low_anomaly_timestamp as ISO Z string; keep None if no timestamp
summary_df['earliest_low_anomaly_timestamp'] = pd.to_datetime(summary_df['earliest_low_anomaly_timestamp'])
summary_df['earliest_low_anomaly_timestamp'] = summary_df['earliest_low_anomaly_timestamp'].dt.strftime('%Y-%m-%dT%H:%M:%SZ')
summary_df.loc[summary_df['earliest_low_anomaly_timestamp'].isna(), 'earliest_low_anomaly_timestamp'] = None

# 4) Sort by low_anomaly_count desc then high_anomaly_count desc, limit to top 20
result_top20 = (
    summary_df.sort_values(['low_anomaly_count','high_anomaly_count'], ascending=[False, False])
    .loc[:, ['cmdb_id','trace_name','global_p95','global_p05','row_count_in_window',
             'high_anomaly_count','low_anomaly_count','earliest_low_anomaly_timestamp',
             'min_value_in_window','max_value_in_window']]
    .head(20)
)

# 5) Short summary: cmdb_id with any low row_count anomalies (trace_name contains 'row_count')
rowcount_low = (
    summary_df[summary_df['trace_name'].str.contains('row_count', na=False) & (summary_df['low_anomaly_count']>0)]
    .groupby('cmdb_id', dropna=False)['low_anomaly_count']
    .sum()
    .reset_index(name='total_low_anomalies')
    .sort_values('total_low_anomalies', ascending=False)
)

if rowcount_low.shape[0] == 0:
    summary_line = "No low 'row_count' anomalies detected in the window for any cmdb_id."
else:
    parts = [f"{r.cmdb_id}:{r.total_low_anomalies}" for r in rowcount_low.itertuples()]
    summary_line = "Low 'row_count' anomalies by cmdb_id: " + ", ".join(parts)

# Display compact outputs (DataFrame and summary)
result_top20, rowcount_low, summary_line

```
Out[3]:
```
Summary (trace.csv, global thresholds computed on full data; window 2024-01-20 07:13:22–07:43:22 UTC):

- Global P95 and P05 were computed across the entire trace dataset before any time filtering.
- Within the 30-minute window many trace series show both low and high anomalies (value < global_p05 and value > global_p95). The top rows are dominated by checkoutservice trace series (duration_mean, duration_p95, row_count and also traces to/from frontend, currency, email, payment). For those top checkoutservice series each had 25 rows in the window, high_anomaly_count = 2 and low_anomaly_count = 2. Earliest low-anomaly timestamps for checkoutservice row_count/duration series appear around 2024-01-20T07:16:00Z–07:25:00Z; min/max values in-window are close to but on both sides of the global thresholds.
- In short, the most affected trace source is checkoutservice (multiple trace.* metrics with both low and high anomalies during the window).

Low 'row_count' anomalies summary (cmdb_id : total low_anomaly_count), sorted:
- checkoutservice: 16
- frontendservice: 14
- root: 14
- recommendationservice: 10
- productcatalogservice: 8
- currencyservice: 6
- emailservice: 4
- paymentservice: 4

Conclusion: checkoutservice shows the largest number of low trace row_count anomalies in the window, with frontendservice and root also notably affected. Multiple trace series across services exhibited both unusually low and unusually high values relative to their global P05/P95.

The original code execution output of IPython Kernel is also provided below for reference:

(            cmdb_id                                trace_name  global_p95  global_p05  row_count_in_window  high_anomaly_count  low_anomaly_count earliest_low_anomaly_timestamp  min_value_in_window  max_value_in_window
0   checkoutservice  trace.from_checkoutservice.duration_mean    0.029062    0.003680                   25                   2                  2           2024-01-20T07:18:00Z             0.003649             0.030161
1   checkoutservice   trace.from_checkoutservice.duration_p95    0.204455    0.005469                   25                   2                  2           2024-01-20T07:25:00Z             0.005256             0.204713
3   checkoutservice      trace.from_checkoutservice.row_count  311.800000  126.400000                   25                   2                  2           2024-01-20T07:16:00Z            80.000000           322.000000
4   checkoutservice  trace.from_frontendservice.duration_mean    0.308767    0.084400                   25                   2                  2           2024-01-20T07:16:00Z             0.082806             0.311114
5   checkoutservice   trace.from_frontendservice.duration_p95    0.406156    0.132107                   25                   2                  2           2024-01-20T07:22:00Z             0.124271             0.407442
7   checkoutservice      trace.from_frontendservice.row_count   36.800000   15.600000                   25                   2                  2           2024-01-20T07:16:00Z            10.000000            37.000000
8   checkoutservice             trace.from_root.duration_mean    0.018101    0.012928                   25                   2                  2           2024-01-20T07:25:00Z             0.012158             0.018491
9   checkoutservice              trace.from_root.duration_p95    0.073140    0.059773                   25                   2                  2           2024-01-20T07:28:00Z             0.058204             0.073642
11  checkoutservice                 trace.from_root.row_count  154.800000   66.000000                   25                   2                  2           2024-01-20T07:16:00Z            38.000000           161.000000
12  checkoutservice    trace.to_checkoutservice.duration_mean    0.029062    0.003680                   25                   2                  2           2024-01-20T07:18:00Z             0.003649             0.030161
13  checkoutservice     trace.to_checkoutservice.duration_p95    0.204455    0.005469                   25                   2                  2           2024-01-20T07:25:00Z             0.005256             0.204713
15  checkoutservice        trace.to_checkoutservice.row_count  311.800000  126.400000                   25                   2                  2           2024-01-20T07:16:00Z            80.000000           322.000000
16  checkoutservice    trace.to_currencyservice.duration_mean    0.002765    0.000150                   25                   2                  2           2024-01-20T07:19:00Z             0.000147             0.002999
17  checkoutservice     trace.to_currencyservice.duration_p95    0.001522    0.000206                   25                   2                  2           2024-01-20T07:26:00Z             0.000201             0.021227
19  checkoutservice        trace.to_currencyservice.row_count  130.800000   48.400000                   25                   2                  2           2024-01-20T07:16:00Z            30.000000           137.000000
20  checkoutservice       trace.to_emailservice.duration_mean    0.000334    0.000251                   25                   2                  2           2024-01-20T07:20:00Z             0.000235             0.000365
21  checkoutservice        trace.to_emailservice.duration_p95    0.000770    0.000354                   25                   2                  2           2024-01-20T07:20:00Z             0.000316             0.000811
23  checkoutservice           trace.to_emailservice.row_count   36.800000   15.600000                   25                   2                  2           2024-01-20T07:16:00Z            10.000000            37.000000
24  checkoutservice     trace.to_paymentservice.duration_mean    0.000329    0.000204                   25                   2                  2           2024-01-20T07:18:00Z             0.000200             0.000564
25  checkoutservice      trace.to_paymentservice.duration_p95    0.000970    0.000277                   25                   2                  2           2024-01-20T07:19:00Z             0.000257             0.001573,                  cmdb_id  total_low_anomalies
0        checkoutservice                   16
3        frontendservice                   14
7                   root                   14
6  recommendationservice                   10
5  productcatalogservice                    8
1        currencyservice                    6
2           emailservice                    4
4         paymentservice                    4, "Low 'row_count' anomalies by cmdb_id: checkoutservice:16, frontendservice:14, root:14, recommendationservice:10, productcatalogservice:8, currencyservice:6, emailservice:4, paymentservice:4")```
```

In [ ]:
# In[4]:

import pandas as pd
from pandas.api.types import is_datetime64_any_dtype, is_integer_dtype

# Reuse existing trace_df and metric_df if present; otherwise load
try:
    trace_df
except NameError:
    trace_df = pd.read_csv('trace.csv')

try:
    metric_df
except NameError:
    metric_df = pd.read_csv('metric.csv')

# Ensure timestamps are parsed to UTC datetimes (unit='s')
for df in (trace_df, metric_df):
    if not is_datetime64_any_dtype(df['timestamp']):
        if is_integer_dtype(df['timestamp']):
            df['timestamp'] = pd.to_datetime(df['timestamp'], unit='s', utc=True)
        else:
            df['timestamp'] = pd.to_datetime(df['timestamp'], utc=True)

# Ensure numeric values
trace_df['value'] = pd.to_numeric(trace_df['value'], errors='coerce')
metric_df['value'] = pd.to_numeric(metric_df['value'], errors='coerce')

# Incident window boundaries (UTC)
start_ts = pd.to_datetime('2024-01-20 07:13:22', utc=True)
end_ts   = pd.to_datetime('2024-01-20 07:43:22', utc=True)

# 1) Trace: compute global P05 per (cmdb_id, trace_name) on full trace_df
trace_global_p05 = (
    trace_df.groupby(['cmdb_id','trace_name'], dropna=False)['value']
    .agg(global_p05=lambda x: x.quantile(0.05))
    .reset_index()
)

# Filter trace_df to window
trace_window = trace_df[(trace_df['timestamp'] >= start_ts) & (trace_df['timestamp'] <= end_ts)].copy()

# Merge global_p05 into window rows
trace_window = trace_window.merge(trace_global_p05, on=['cmdb_id','trace_name'], how='left')

# Select rows where trace_name contains 'row_count' AND value < global_p05
trace_low_rowcount_events = trace_window[
    trace_window['trace_name'].str.contains('row_count', na=False) &
    (trace_window['value'] < trace_window['global_p05'])
].copy()

# Format timestamp as ISO UTC string and select required columns, sort ascending, limit top 30
trace_low_rowcount_events['timestamp'] = trace_low_rowcount_events['timestamp'].dt.strftime('%Y-%m-%dT%H:%M:%SZ')
trace_low_rowcount_events = trace_low_rowcount_events.loc[
    :, ['timestamp','cmdb_id','trace_name','value']
].sort_values('timestamp', ascending=True).head(30)

# 2) Metric: compute global P95 per (cmdb_id, kpi_name) on full metric_df
metric_global_p95 = (
    metric_df.groupby(['cmdb_id','kpi_name'], dropna=False)['value']
    .agg(global_p95=lambda x: x.quantile(0.95))
    .reset_index()
)

# Filter metric_df to window
metric_window = metric_df[(metric_df['timestamp'] >= start_ts) & (metric_df['timestamp'] <= end_ts)].copy()

# Merge global_p95 into window rows
metric_window = metric_window.merge(metric_global_p95, on=['cmdb_id','kpi_name'], how='left')

# Select rows where kpi_name == 'mem' AND value > global_p95
mem_high_events = metric_window[
    (metric_window['kpi_name'] == 'mem') &
    (metric_window['value'] > metric_window['global_p95'])
].copy()

# Format timestamp as ISO UTC string and select required columns, sort ascending, limit top 30
mem_high_events['timestamp'] = mem_high_events['timestamp'].dt.strftime('%Y-%m-%dT%H:%M:%SZ')
mem_high_events = mem_high_events.loc[
    :, ['timestamp','cmdb_id','kpi_name','value']
].sort_values('timestamp', ascending=True).head(30)

# Display the two result DataFrames (compact)
trace_low_rowcount_events, mem_high_events

```
Out[4]:
```
Summary of findings (global thresholds computed on full files; window 2024-01-20 07:13:22–07:43:22 UTC):

- Trace low "row_count" events:
  - Extract returned 30 rows (top 30 in the window) of trace series where trace_name contains "row_count" and value < global P05.
  - Earliest low row_count timestamps in the list occur at 2024-01-20T07:16:00Z (examples: checkoutservice.trace.from_checkoutservice.row_count = 120.0, paymentservice.trace.from_root.row_count = 8.0, productcatalogservice.trace.from_frontendservice.row_count = 2302.0, frontendservice.trace.to_productcatalogservice.row_count = 2302.0, etc.).
  - Many services show low row_count values around 07:16:00Z.

- Metric mem high events:
  - Extract returned 22 rows (mem series where value > global P95 in the same window).
  - Earliest mem anomaly is at 2024-01-20T07:18:00Z for cartservice (mem = 4.45825e7). Other mem spikes follow at 07:19 (adservice), 07:21 (redis), 07:23 (checkoutservice), and continue through ~07:39–07:40 for several services.

- Temporal relationship and inference:
  - Low trace row_count events start around 07:16:00Z, about ~2 minutes before the first observed mem spike at 07:18:00Z.
  - Multiple services show low row_count at ~07:16, while memory spikes appear shortly afterward across several services.
  - This temporal ordering suggests that reduced trace row_counts (a drop in observed request counts/propagation) preceded memory anomalies. That ordering is consistent with a scenario where traffic/trace propagation changed first and memory behavior changed afterwards — but it does not prove causation by itself. Further investigation of logs, tracing spans, and resource graphs is needed to confirm root cause.

- Suggested next steps:
  - Inspect service logs (error_logs/logs) around 07:15–07:25 for checkoutservice and frontendservice for errors/timeouts.
  - Examine traces to see whether a frontend or network issue caused a drop in cross-service calls at ~07:16.
  - Check memory usage patterns and GC/heap activity on the services with mem spikes (adservice, cartservice, checkoutservice, productcatalogservice, recommendationservice, etc.) to determine why memory rose after the row_count drop.

If you want, I can display the earliest few rows from each extracted list again (e.g., head(10)) for quick inspection.

The original code execution output of IPython Kernel is also provided below for reference:

(               timestamp                cmdb_id                                  trace_name   value
3   2024-01-20T07:16:00Z        checkoutservice        trace.from_checkoutservice.row_count   120.0
87  2024-01-20T07:16:00Z         paymentservice                   trace.from_root.row_count     8.0
91  2024-01-20T07:16:00Z  productcatalogservice        trace.from_checkoutservice.row_count    30.0
95  2024-01-20T07:16:00Z  productcatalogservice        trace.from_frontendservice.row_count  2302.0
99  2024-01-20T07:16:00Z  productcatalogservice  trace.from_recommendationservice.row_count   353.0
..                   ...                    ...                                         ...     ...
75  2024-01-20T07:16:00Z        frontendservice    trace.to_productcatalogservice.row_count  2302.0
27  2024-01-20T07:16:00Z        checkoutservice           trace.to_paymentservice.row_count    15.0
31  2024-01-20T07:16:00Z        checkoutservice    trace.to_productcatalogservice.row_count    30.0
35  2024-01-20T07:16:00Z        currencyservice        trace.from_checkoutservice.row_count    45.0
23  2024-01-20T07:16:00Z        checkoutservice             trace.to_emailservice.row_count    15.0

[30 rows x 4 columns],                  timestamp                cmdb_id kpi_name         value
159   2024-01-20T07:18:00Z            cartservice      mem  4.458250e+07
226   2024-01-20T07:19:00Z              adservice      mem  1.094922e+08
436   2024-01-20T07:21:00Z                  redis      mem  2.999637e+06
529   2024-01-20T07:23:00Z            cartservice      mem  4.460107e+07
535   2024-01-20T07:23:00Z        checkoutservice      mem  1.222117e+07
...                    ...                    ...      ...           ...
1719  2024-01-20T07:39:00Z        checkoutservice      mem  1.201295e+07
1726  2024-01-20T07:39:00Z        currencyservice      mem  4.552335e+07
1733  2024-01-20T07:39:00Z           emailservice      mem  4.375818e+07
1830  2024-01-20T07:40:00Z  productcatalogservice      mem  1.131012e+07
1837  2024-01-20T07:40:00Z  recommendationservice      mem  4.562285e+07

[22 rows x 4 columns])

**Note**: The printed pandas DataFrame is truncated due to its size. Only **10 rows** are displayed, which may introduce observation bias due to the incomplete table. If you want to comprehensively understand the details without bias, please ask Executor using `df.head(X)` to display more rows.```
```

In [ ]:
# In[5]:

import pandas as pd
from pandas.api.types import is_datetime64_any_dtype, is_integer_dtype

# Reuse existing dataframes if present, otherwise load
try:
    log_df
except NameError:
    log_df = pd.read_csv('log.csv')
try:
    error_df
except NameError:
    error_df = pd.read_csv('error_logs.csv')

# Ensure timestamps are parsed to UTC datetimes (unit='s')
for df in (log_df, error_df):
    if not is_datetime64_any_dtype(df['timestamp']):
        if is_integer_dtype(df['timestamp']):
            df['timestamp'] = pd.to_datetime(df['timestamp'], unit='s', utc=True)
        else:
            df['timestamp'] = pd.to_datetime(df['timestamp'], utc=True)

# Ensure numeric 'value' in log_df
log_df['value'] = pd.to_numeric(log_df['value'], errors='coerce')

# Incident window (UTC)
start_ts = pd.to_datetime('2024-01-20 07:13:22', utc=True)
end_ts   = pd.to_datetime('2024-01-20 07:43:22', utc=True)

# 1) Compute global P95 and P05 per (cmdb_id, log_name) on full log_df (global before filtering)
global_log_thresh = (
    log_df.groupby(['cmdb_id','log_name'], dropna=False)['value']
    .agg(global_p95=lambda x: x.quantile(0.95),
         global_p05=lambda x: x.quantile(0.05))
    .reset_index()
)

# Candidate cmdb_id list
candidates = [
    "checkoutservice","frontendservice","adservice","cartservice","paymentservice",
    "productcatalogservice","recommendationservice","currencyservice","emailservice","redis"
]

# Filter log_df to window and to candidate cmdb_ids
log_window = log_df[
    (log_df['timestamp'] >= start_ts) & (log_df['timestamp'] <= end_ts) & (log_df['cmdb_id'].isin(candidates))
].copy()

# Merge global thresholds
log_window = log_window.merge(global_log_thresh, on=['cmdb_id','log_name'], how='left')

# Select rows where:
# (a) log_name == 'log.error_count' and value > global_p95
# OR
# (b) log_name == 'log.row_count' and value < global_p05
cond_error_high = (log_window['log_name'] == 'log.error_count') & (log_window['value'] > log_window['global_p95'])
cond_rowcount_low = (log_window['log_name'] == 'log.row_count') & (log_window['value'] < log_window['global_p05'])
log_anomalies = log_window[cond_error_high | cond_rowcount_low].copy()

# Format timestamp as ISO UTC and select columns, sort ascending, limit top 50
log_anomalies['timestamp'] = log_anomalies['timestamp'].dt.strftime('%Y-%m-%dT%H:%M:%SZ')
log_anomalies = log_anomalies.loc[:, ['timestamp','cmdb_id','log_name','value']].sort_values('timestamp', ascending=True).head(50)

# 2) Filter error_logs to window and return up to earliest 20 rows (timestamp ISO UTC, cmdb_id, message)
error_window = error_df[(error_df['timestamp'] >= start_ts) & (error_df['timestamp'] <= end_ts)].copy()
if error_window.shape[0] == 0:
    error_window_result = "No error_log rows in the incident window."
else:
    error_window['timestamp'] = error_window['timestamp'].dt.strftime('%Y-%m-%dT%H:%M:%SZ')
    error_window_result = error_window.loc[:, ['timestamp','cmdb_id','message']].sort_values('timestamp', ascending=True).head(20)

# Display compact results
log_anomalies, error_window_result

```
Out[5]:
```
Summary of log and error-log findings (window 2024-01-20 07:13:22–07:43:22 UTC):

- Detected low log.row_count events (value < global P05) at 07:16:00Z across multiple services:  
  adservice (289), cartservice (481), checkoutservice (45), currencyservice (1485), emailservice (15), paymentservice (30), recommendationservice (353).  
  A similar set of low row_count events appears again near 07:40:00Z for many of the same services (e.g., adservice 157, cartservice 271, checkoutservice 30, currencyservice 895, recommendationservice 190).

- No log.error_count anomalies were found (no rows where log.error_count > global P95).

- error_logs.csv contained no entries in the incident window.

Interpretation / next step suggestion:
- There was a coordinated drop in log.row_count across many services around 07:16:00Z (and another dip near 07:40:00Z). This aligns with the trace/metric observations you previously ran (trace row_count drops around 07:16 followed by memory spikes).  
- Because there are no error_log entries and no high log.error_count, investigate upstream traffic or routing changes (frontend or network) and compare traces/timestamps across services to determine whether a traffic reduction or propagation issue preceded the resource anomalies.

The original code execution output of IPython Kernel is also provided below for reference:

(                timestamp                cmdb_id       log_name  value
1    2024-01-20T07:16:00Z              adservice  log.row_count    289
3    2024-01-20T07:16:00Z            cartservice  log.row_count    481
5    2024-01-20T07:16:00Z        checkoutservice  log.row_count     45
7    2024-01-20T07:16:00Z        currencyservice  log.row_count   1485
9    2024-01-20T07:16:00Z           emailservice  log.row_count     15
11   2024-01-20T07:16:00Z         paymentservice  log.row_count     30
13   2024-01-20T07:16:00Z  recommendationservice  log.row_count    353
347  2024-01-20T07:40:00Z              adservice  log.row_count    157
349  2024-01-20T07:40:00Z            cartservice  log.row_count    271
351  2024-01-20T07:40:00Z        checkoutservice  log.row_count     30
353  2024-01-20T07:40:00Z        currencyservice  log.row_count    895
355  2024-01-20T07:40:00Z           emailservice  log.row_count     10
357  2024-01-20T07:40:00Z         paymentservice  log.row_count     20
359  2024-01-20T07:40:00Z  recommendationservice  log.row_count    190, 'No error_log rows in the incident window.')```
```